In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import zipfile
import time

PROJECT = "/content/ai_voice_project"
os.makedirs(PROJECT, exist_ok=True)

datasets = {
    "Dataset1": "/content/drive/MyDrive/Dataset1.zip",
    "Dataset2": "/content/drive/MyDrive/Dataset2.zip",
    "Dataset3": "/content/drive/MyDrive/Dataset3.zip",
}

print("=" * 70)
print("ROZBAĽOVANIE DATASETOV")
print("=" * 70)

for name, zip_path in datasets.items():

    destination = os.path.join(PROJECT, name)
    os.makedirs(destination, exist_ok=True)

    print(f"\n📦 {name}")
    print(f"   ZIP: {zip_path}")
    print(f"   → {destination}")

    start = time.time()

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(destination)

    elapsed = time.time() - start

    print(f"   ✅ Hotovo za {elapsed:.1f} s")

print("\n" + "=" * 70)
print("✅ Všetky datasety rozbalené.")

ROZBAĽOVANIE DATASETOV

📦 Dataset1
   ZIP: /content/drive/MyDrive/Dataset1.zip
   → /content/ai_voice_project/Dataset1
   ✅ Hotovo za 33.2 s

📦 Dataset2
   ZIP: /content/drive/MyDrive/Dataset2.zip
   → /content/ai_voice_project/Dataset2
   ✅ Hotovo za 3.9 s

📦 Dataset3
   ZIP: /content/drive/MyDrive/Dataset3.zip
   → /content/ai_voice_project/Dataset3
   ✅ Hotovo za 46.9 s

✅ Všetky datasety rozbalené.


In [ ]:
import torch
import transformers
from transformers import pipeline

MODEL_ID = "Speech-Arena-2025/DF_Arena_1B_V_1"

print("=" * 70)
print("DF ARENA 1B V1")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("\n⏳ Načítavam DF Arena 1B...")

pipe_df_arena = pipeline(
    "antispoofing",
    model=MODEL_ID,
    trust_remote_code=True,
    device="cuda"
)

print("\n✅ DF Arena 1B úspešne načítaný.")

DF ARENA 1B V1
PyTorch: 2.11.0+cu130
Transformers: 5.18.0
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB

⏳ Načítavam DF Arena 1B...


config.json:   0%|          | 0.00/600 [00:00<?, ?B/s]

configuration_antispoofing.py:   0%|          | 0.00/325 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/Speech-Arena-2025/DF_Arena_1B_V_1:
- configuration_antispoofing.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pipeline_antispoofing.py:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

feature_extraction_antispoofing.py:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/966 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


✅ DF Arena 1B úspešne načítaný.


In [ ]:
import os
import glob
import csv
import time
import librosa
import torch

# ---------------------------------------------------------
# CESTY
# ---------------------------------------------------------

fake_files = sorted(
    glob.glob(
        "/content/ai_voice_project/Dataset3/**/Fake_Audios/*.wav",
        recursive=True
    )
)

real_files = sorted(
    glob.glob(
        "/content/ai_voice_project/Dataset3/**/Real_Audios/*.wav",
        recursive=True
    )
)

OUTPUT = "/content/drive/MyDrive/df_arena_dataset3_scores.csv"

samples = (
    [("FAKE", 1, p) for p in fake_files] +
    [("REAL", 0, p) for p in real_files]
)

print("FAKE :", len(fake_files))
print("REAL :", len(real_files))
print("TOTAL:", len(samples))
print("CSV  :", OUTPUT)


# ---------------------------------------------------------
# RESUME – ak Colab spadne, pokračujeme kde sme skončili
# ---------------------------------------------------------

processed = set()

if os.path.exists(OUTPUT):
    with open(OUTPUT, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            processed.add(row["path"])

    print("\n🔄 Existujúci CSV nájdený.")
    print("Už spracovaných:", len(processed))

else:
    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "dataset",
            "file",
            "path",
            "label",
            "class",
            "prediction",
            "score_spoof",
            "score_bonafide",
            "duration_sec",
            "correct"
        ])

    print("\n🆕 Vytvorený nový CSV.")


# ---------------------------------------------------------
# INFERENCE
# ---------------------------------------------------------

start_all = time.time()
new_count = 0
errors = 0

for i, (true_class, label, path) in enumerate(samples, 1):

    if path in processed:
        continue

    try:
        audio, sr = librosa.load(
            path,
            sr=16000,
            mono=True
        )

        duration = len(audio) / sr

        with torch.inference_mode():
            result = pipe_df_arena(audio)

        prediction = result["label"]

        score_spoof = float(
            result["all_scores"]["spoof"]
        )

        score_bonafide = float(
            result["all_scores"]["bonafide"]
        )

        predicted_class = (
            "FAKE"
            if prediction == "spoof"
            else "REAL"
        )

        correct = int(predicted_class == true_class)

        # ---------------------------------------------
        # OKAMŽITÉ ULOŽENIE NA GOOGLE DRIVE
        # ---------------------------------------------

        with open(
            OUTPUT,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(f)

            writer.writerow([
                "Dataset3",
                os.path.basename(path),
                path,
                label,
                true_class,
                prediction,
                score_spoof,
                score_bonafide,
                duration,
                correct
            ])

        new_count += 1

        # priebežný výpis
        if new_count % 10 == 0 or new_count == 1:

            elapsed = time.time() - start_all

            print(
                f"[{i}/{len(samples)}] "
                f"nových={new_count} | "
                f"{os.path.basename(path)} | "
                f"{true_class} → {predicted_class} | "
                f"spoof={score_spoof:.4f} | "
                f"{duration:.1f}s | "
                f"čas={elapsed/60:.1f} min"
            )

    except Exception as e:

        errors += 1

        print(
            f"\n❌ ERROR [{i}/{len(samples)}]"
        )
        print(path)
        print(type(e).__name__, str(e))

        # uvoľnenie CUDA cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("\n" + "=" * 80)
print("HOTOVO")
print("=" * 80)

print("Nových spracovaných:", new_count)
print("Chyby:", errors)
print(
    "Čas:",
    round((time.time() - start_all) / 60, 2),
    "min"
)
print("Výsledky:", OUTPUT)

FAKE : 431
REAL : 423
TOTAL: 854
CSV  : /content/drive/MyDrive/df_arena_dataset3_scores.csv

🆕 Vytvorený nový CSV.
[1/854] nových=1 | fake_001.wav | FAKE → FAKE | spoof=0.9989 | 10.4s | čas=0.0 min


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[10/854] nových=10 | fake_010.wav | FAKE → FAKE | spoof=0.9994 | 11.7s | čas=0.0 min
[20/854] nových=20 | fake_020.wav | FAKE → FAKE | spoof=0.9982 | 11.4s | čas=0.1 min
[30/854] nových=30 | fake_030.wav | FAKE → FAKE | spoof=0.8394 | 12.8s | čas=0.1 min
[40/854] nových=40 | fake_040.wav | FAKE → FAKE | spoof=0.9997 | 29.3s | čas=0.2 min
[50/854] nových=50 | fake_050.wav | FAKE → FAKE | spoof=0.9968 | 11.6s | čas=0.2 min
[60/854] nových=60 | fake_060.wav | FAKE → FAKE | spoof=0.9287 | 13.2s | čas=0.2 min
[70/854] nových=70 | fake_070.wav | FAKE → FAKE | spoof=0.9981 | 11.0s | čas=0.3 min
[80/854] nových=80 | fake_080.wav | FAKE → FAKE | spoof=0.9997 | 12.6s | čas=0.3 min
[90/854] nových=90 | fake_090.wav | FAKE → FAKE | spoof=0.6982 | 29.0s | čas=0.4 min
[100/854] nových=100 | fake_100.wav | FAKE → FAKE | spoof=0.9863 | 23.9s | čas=0.4 min
[110/854] nových=110 | fake_110.wav | FAKE → FAKE | spoof=0.9998 | 50.0s | čas=0.5 min
[120/854] nových=120 | fake_120.wav | FAKE → FAKE | spoof=0.9

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

CSV = "/content/drive/MyDrive/df_arena_dataset3_scores.csv"

df = pd.read_csv(CSV)

print("=" * 75)
print("DF ARENA 1B V1 — DATASET3")
print("=" * 75)

print("\nPočet vzoriek:", len(df))
print(df["class"].value_counts())


# ---------------------------------------------------------
# LABELY
# 1 = FAKE
# 0 = REAL
# ---------------------------------------------------------

y_true = df["label"].astype(int).values
score_fake = df["score_spoof"].astype(float).values

# threshold 0.5
y_pred = (score_fake >= 0.5).astype(int)


# ---------------------------------------------------------
# METRIKY
# ---------------------------------------------------------

accuracy = accuracy_score(y_true, y_pred)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

auc = roc_auc_score(
    y_true,
    score_fake
)

precision_fake = precision_score(
    y_true,
    y_pred,
    pos_label=1
)

recall_fake = recall_score(
    y_true,
    y_pred,
    pos_label=1
)

f1_fake = f1_score(
    y_true,
    y_pred,
    pos_label=1
)

recall_real = recall_score(
    y_true,
    y_pred,
    pos_label=0
)


# ---------------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


# ---------------------------------------------------------
# VÝSLEDKY
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("HLAVNÉ METRIKY")
print("=" * 75)

print(f"Accuracy          : {accuracy*100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc*100:.2f}%")
print(f"ROC AUC           : {auc:.4f}")

print("\nFAKE:")
print(f"Precision : {precision_fake*100:.2f}%")
print(f"Recall    : {recall_fake*100:.2f}%")
print(f"F1        : {f1_fake:.4f}")

print("\nREAL:")
print(f"Recall    : {recall_real*100:.2f}%")


print("\n" + "=" * 75)
print("CONFUSION MATRIX")
print("=" * 75)

print()
print("                 PRED REAL    PRED FAKE")
print(f"TRUE REAL        {tn:9d}    {fp:9d}")
print(f"TRUE FAKE        {fn:9d}    {tp:9d}")


print("\nSprávne REAL :", tn, "/", tn + fp)
print("Falošne FAKE :", fp, "/", tn + fp)

print("\nSprávne FAKE :", tp, "/", tp + fn)
print("Falošne REAL :", fn, "/", tp + fn)


# ---------------------------------------------------------
# DISTRIBÚCIA SCORE
# ---------------------------------------------------------

print("\n" + "=" * 75)
print("SCORE_SPOOF DISTRIBÚCIA")
print("=" * 75)

for cls in ["REAL", "FAKE"]:

    scores = df.loc[
        df["class"] == cls,
        "score_spoof"
    ]

    print(f"\n{cls}")
    print(f"mean   : {scores.mean():.6f}")
    print(f"median : {scores.median():.6f}")
    print(f"min    : {scores.min():.6f}")
    print(f"max    : {scores.max():.6f}")


# ---------------------------------------------------------
# NAJISTEJŠIE CHYBY
# ---------------------------------------------------------

df["predicted"] = np.where(
    df["score_spoof"] >= 0.5,
    "FAKE",
    "REAL"
)

errors = df[
    df["class"] != df["predicted"]
].copy()

print("\n" + "=" * 75)
print("CHYBY")
print("=" * 75)

print("Celkový počet chýb:", len(errors))

false_fake = errors[
    errors["class"] == "REAL"
].sort_values(
    "score_spoof",
    ascending=False
)

false_real = errors[
    errors["class"] == "FAKE"
].sort_values(
    "score_spoof",
    ascending=True
)

print("\nNajistejšie REAL → FAKE chyby:")

print(
    false_fake[
        [
            "file",
            "score_spoof",
            "duration_sec"
        ]
    ].head(10).to_string(index=False)
)

print("\nNajistejšie FAKE → REAL chyby:")

print(
    false_real[
        [
            "file",
            "score_spoof",
            "duration_sec"
        ]
    ].head(10).to_string(index=False)
)

DF ARENA 1B V1 — DATASET3

Počet vzoriek: 854
class
FAKE    431
REAL    423
Name: count, dtype: int64

HLAVNÉ METRIKY
Accuracy          : 88.88%
Balanced Accuracy : 88.81%
ROC AUC           : 0.9591

FAKE:
Precision : 84.01%
Recall    : 96.29%
F1        : 0.8973

REAL:
Recall    : 81.32%

CONFUSION MATRIX

                 PRED REAL    PRED FAKE
TRUE REAL              344           79
TRUE FAKE               16          415

Správne REAL : 344 / 423
Falošne FAKE : 79 / 423

Správne FAKE : 415 / 431
Falošne REAL : 16 / 431

SCORE_SPOOF DISTRIBÚCIA

REAL
mean   : 0.282772
median : 0.190800
min    : 0.008300
max    : 0.999806

FAKE
mean   : 0.945548
median : 0.998311
min    : 0.062090
max    : 0.999919

CHYBY
Celkový počet chýb: 95

Najistejšie REAL → FAKE chyby:
        file  score_spoof  duration_sec
real_038.wav     0.999806      6.272000
real_093.wav     0.999786      6.058687
real_019.wav     0.999764      6.656000
real_048.wav     0.999726      5.418687
real_080.wav     0.999716    

In [ ]:
import os
import glob
import csv
import time
import librosa
import torch

# ============================================================
# DATASET1
# ============================================================

ROOT = "/content/ai_voice_project/Dataset1"
OUTPUT = "/content/drive/MyDrive/df_arena_dataset1_scores.csv"

AUDIO_EXTS = ["wav", "mp3", "m4a", "flac", "ogg", "aac"]

all_files = []

for ext in AUDIO_EXTS:
    all_files += glob.glob(
        os.path.join(ROOT, "**", f"*.{ext}"),
        recursive=True
    )
    all_files += glob.glob(
        os.path.join(ROOT, "**", f"*.{ext.upper()}"),
        recursive=True
    )

all_files = sorted(set(all_files))

# Triedu určíme podľa priečinka Fake / Real
fake_files = [
    f for f in all_files
    if "fake" in f.lower()
]

real_files = [
    f for f in all_files
    if "real" in f.lower()
]

print("=" * 75)
print("DF ARENA 1B — DATASET1")
print("=" * 75)

print("FAKE :", len(fake_files))
print("REAL :", len(real_files))
print("TOTAL:", len(fake_files) + len(real_files))
print("CSV  :", OUTPUT)

# Očakávame 21 + 21
assert len(fake_files) == 21, (
    f"❌ Očakávali sme 21 FAKE, našlo sa {len(fake_files)}"
)

assert len(real_files) == 21, (
    f"❌ Očakávali sme 21 REAL, našlo sa {len(real_files)}"
)

samples = (
    [("FAKE", 1, p) for p in fake_files] +
    [("REAL", 0, p) for p in real_files]
)


# ============================================================
# RESUME
# ============================================================

processed = set()

if os.path.exists(OUTPUT):

    with open(OUTPUT, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            processed.add(row["path"])

    print("\n🔄 Existujúci CSV nájdený.")
    print("Už spracovaných:", len(processed))

else:

    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "dataset",
            "file",
            "path",
            "label",
            "class",
            "prediction",
            "score_spoof",
            "score_bonafide",
            "duration_sec",
            "correct"
        ])

    print("\n🆕 Vytvorený nový CSV.")


# ============================================================
# INFERENCE
# ============================================================

start = time.time()
new_count = 0
errors = 0

for i, (true_class, label, path) in enumerate(samples, 1):

    if path in processed:
        continue

    try:

        audio, sr = librosa.load(
            path,
            sr=16000,
            mono=True
        )

        duration = len(audio) / sr

        with torch.inference_mode():
            result = pipe_df_arena(audio)

        prediction = result["label"]

        score_spoof = float(
            result["all_scores"]["spoof"]
        )

        score_bonafide = float(
            result["all_scores"]["bonafide"]
        )

        predicted_class = (
            "FAKE"
            if prediction == "spoof"
            else "REAL"
        )

        correct = int(
            predicted_class == true_class
        )


        # ====================================================
        # OKAMŽITÉ ULOŽENIE
        # ====================================================

        with open(
            OUTPUT,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(f)

            writer.writerow([
                "Dataset1",
                os.path.basename(path),
                path,
                label,
                true_class,
                prediction,
                score_spoof,
                score_bonafide,
                duration,
                correct
            ])

        new_count += 1

        print(
            f"[{i:02d}/{len(samples)}] "
            f"{os.path.basename(path)} | "
            f"{true_class} → {predicted_class} | "
            f"spoof={score_spoof:.4f} | "
            f"{duration:.1f}s"
        )

    except Exception as e:

        errors += 1

        print("\n❌ ERROR")
        print(path)
        print(type(e).__name__, str(e))

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("\n" + "=" * 75)
print("HOTOVO")
print("=" * 75)

print("Nových spracovaných:", new_count)
print("Chyby:", errors)
print(
    "Čas:",
    round((time.time() - start) / 60, 2),
    "min"
)
print("Výsledky:", OUTPUT)

DF ARENA 1B — DATASET1
FAKE : 21
REAL : 21
TOTAL: 42
CSV  : /content/drive/MyDrive/df_arena_dataset1_scores.csv

🆕 Vytvorený nový CSV.
[01/42] Recording (1).wav | FAKE → REAL | spoof=0.2689 | 494.9s
[02/42] Recording (10).wav | FAKE → REAL | spoof=0.3715 | 317.5s
[03/42] Recording (11).wav | FAKE → REAL | spoof=0.1881 | 337.9s
[04/42] Recording (12).wav | FAKE → FAKE | spoof=0.5428 | 267.8s
[05/42] Recording (13).wav | FAKE → FAKE | spoof=0.7327 | 308.8s
[06/42] Recording (14).wav | FAKE → REAL | spoof=0.4629 | 326.9s
[07/42] Recording (15).wav | FAKE → FAKE | spoof=0.6807 | 325.2s
[08/42] Recording (16).wav | FAKE → FAKE | spoof=0.8281 | 332.0s
[09/42] Recording (17).wav | FAKE → FAKE | spoof=0.5437 | 320.2s
[10/42] Recording (18).wav | FAKE → REAL | spoof=0.3614 | 309.7s
[11/42] Recording (19).wav | FAKE → REAL | spoof=0.4523 | 319.0s
[12/42] Recording (2).wav | FAKE → REAL | spoof=0.4048 | 449.4s
[13/42] Recording (20).wav | FAKE → REAL | spoof=0.4103 | 237.7s
[14/42] Recording (21)

In [ ]:
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

CSV = "/content/drive/MyDrive/df_arena_dataset1_scores.csv"

df1 = pd.read_csv(CSV)

y_true = df1["label"].astype(int).values
score_fake = df1["score_spoof"].astype(float).values

# Rovnaký threshold 0.5
y_pred = (score_fake >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, score_fake)

precision_fake = precision_score(
    y_true, y_pred, pos_label=1
)

recall_fake = recall_score(
    y_true, y_pred, pos_label=1
)

recall_real = recall_score(
    y_true, y_pred, pos_label=0
)

f1_fake = f1_score(
    y_true, y_pred, pos_label=1
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()


print("=" * 75)
print("DF ARENA 1B V1 — DATASET1")
print("=" * 75)

print("Počet vzoriek:", len(df1))

print("\nHLAVNÉ METRIKY")
print("-" * 40)

print(f"Accuracy          : {accuracy*100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc*100:.2f}%")
print(f"ROC AUC           : {auc:.4f}")

print("\nFAKE:")
print(f"Precision : {precision_fake*100:.2f}%")
print(f"Recall    : {recall_fake*100:.2f}%")
print(f"F1        : {f1_fake:.4f}")

print("\nREAL:")
print(f"Recall    : {recall_real*100:.2f}%")

print("\nCONFUSION MATRIX")
print("-" * 40)

print("                 PRED REAL    PRED FAKE")
print(f"TRUE REAL        {tn:9d}    {fp:9d}")
print(f"TRUE FAKE        {fn:9d}    {tp:9d}")

print("\nSprávne REAL :", tn, "/", tn + fp)
print("Falošne FAKE :", fp, "/", tn + fp)

print("Správne FAKE :", tp, "/", tp + fn)
print("Falošne REAL :", fn, "/", tp + fn)


print("\nSCORE_SPOOF DISTRIBÚCIA")
print("-" * 40)

for cls in ["REAL", "FAKE"]:

    scores = df1.loc[
        df1["class"] == cls,
        "score_spoof"
    ]

    print(f"\n{cls}")
    print(f"mean   : {scores.mean():.6f}")
    print(f"median : {scores.median():.6f}")
    print(f"min    : {scores.min():.6f}")
    print(f"max    : {scores.max():.6f}")

DF ARENA 1B V1 — DATASET1
Počet vzoriek: 42

HLAVNÉ METRIKY
----------------------------------------
Accuracy          : 64.29%
Balanced Accuracy : 64.29%
ROC AUC           : 0.9138

FAKE:
Precision : 87.50%
Recall    : 33.33%
F1        : 0.4828

REAL:
Recall    : 95.24%

CONFUSION MATRIX
----------------------------------------
                 PRED REAL    PRED FAKE
TRUE REAL               20            1
TRUE FAKE               14            7

Správne REAL : 20 / 21
Falošne FAKE : 1 / 21
Správne FAKE : 7 / 21
Falošne REAL : 14 / 21

SCORE_SPOOF DISTRIBÚCIA
----------------------------------------

REAL
mean   : 0.118800
median : 0.049852
min    : 0.012042
max    : 0.707792

FAKE
mean   : 0.464085
median : 0.452313
min    : 0.083210
max    : 0.828073


In [ ]:
import os
import glob

ROOT = "/content/ai_voice_project/Dataset2"

print("=" * 80)
print("DATASET2 — KONTROLA ŠTRUKTÚRY")
print("=" * 80)

# ------------------------------------------------------------
# VŠETKY SÚBORY
# ------------------------------------------------------------

all_files = []

for root, dirs, files in os.walk(ROOT):
    for file in files:
        all_files.append(
            os.path.join(root, file)
        )

print("\nCelkový počet všetkých súborov:", len(all_files))


# ------------------------------------------------------------
# AUDIO
# ------------------------------------------------------------

audio_exts = (
    ".wav",
    ".mp3",
    ".m4a",
    ".flac",
    ".ogg",
    ".aac"
)

audio_files = [
    f for f in all_files
    if f.lower().endswith(audio_exts)
]

print("Audio súborov:", len(audio_files))


# ------------------------------------------------------------
# METADATA
# ------------------------------------------------------------

metadata_files = [
    f for f in all_files
    if f.lower().endswith(
        (".xlsx", ".xls", ".csv", ".json")
    )
]

print("\nMetadata súbory:")

for f in metadata_files:
    print("  ", f)


# ------------------------------------------------------------
# UKÁŽKA AUDIO CIEST
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PRVÝCH 30 AUDIO SÚBOROV")
print("=" * 80)

for f in audio_files[:30]:
    print(f)


# ------------------------------------------------------------
# PRIEČINKY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PRIEČINKY")
print("=" * 80)

for root, dirs, files in os.walk(ROOT):

    level = root.replace(ROOT, "").count(os.sep)

    if level <= 4:
        indent = "  " * level
        print(
            f"{indent}{os.path.basename(root)}/"
        )

DATASET2 — KONTROLA ŠTRUKTÚRY

Celkový počet všetkých súborov: 81
Audio súborov: 80

Metadata súbory:
   /content/ai_voice_project/Dataset2/dataset_voice_changing_metainfo.xlsx

PRVÝCH 30 AUDIO SÚBOROV
/content/ai_voice_project/Dataset2/UK/male/2/synthetic_2.mp3
/content/ai_voice_project/Dataset2/UK/male/2/synthetic_1.mp3
/content/ai_voice_project/Dataset2/UK/male/2/original.wav
/content/ai_voice_project/Dataset2/UK/male/2/synthetic_3.mp3
/content/ai_voice_project/Dataset2/UK/male/3/synthetic_2.mp3
/content/ai_voice_project/Dataset2/UK/male/3/synthetic_1.mp3
/content/ai_voice_project/Dataset2/UK/male/3/original.wav
/content/ai_voice_project/Dataset2/UK/male/3/synthetic_3.mp3
/content/ai_voice_project/Dataset2/UK/male/5/synthetic_2.mp3
/content/ai_voice_project/Dataset2/UK/male/5/synthetic_1.mp3
/content/ai_voice_project/Dataset2/UK/male/5/original.wav
/content/ai_voice_project/Dataset2/UK/male/5/synthetic_3.mp3
/content/ai_voice_project/Dataset2/UK/male/1/synthetic_2.mp3
/content/ai_vo

In [ ]:
import os
import glob
import csv
import time
import librosa
import torch

ROOT = "/content/ai_voice_project/Dataset2"
OUTPUT = "/content/drive/MyDrive/df_arena_dataset2_scores.csv"

AUDIO_EXTS = ["wav", "mp3", "m4a", "flac", "ogg", "aac"]

# ============================================================
# NÁJDENIE AUDIO SÚBOROV
# ============================================================

all_files = []

for ext in AUDIO_EXTS:
    all_files += glob.glob(
        os.path.join(ROOT, "**", f"*.{ext}"),
        recursive=True
    )
    all_files += glob.glob(
        os.path.join(ROOT, "**", f"*.{ext.upper()}"),
        recursive=True
    )

all_files = sorted(set(all_files))

real_files = [
    f for f in all_files
    if os.path.basename(f).lower().startswith("original.")
]

fake_files = [
    f for f in all_files
    if os.path.basename(f).lower().startswith("synthetic_")
]

print("=" * 75)
print("DF ARENA 1B — DATASET2")
print("=" * 75)

print("FAKE :", len(fake_files))
print("REAL :", len(real_files))
print("TOTAL:", len(fake_files) + len(real_files))
print("CSV  :", OUTPUT)

# Bezpečnostná kontrola
assert len(fake_files) == 60, (
    f"❌ Očakávali sme 60 FAKE, našlo sa {len(fake_files)}"
)

assert len(real_files) == 20, (
    f"❌ Očakávali sme 20 REAL, našlo sa {len(real_files)}"
)

assert len(fake_files) + len(real_files) == 80

samples = (
    [("FAKE", 1, p) for p in fake_files] +
    [("REAL", 0, p) for p in real_files]
)

# ============================================================
# RESUME
# ============================================================

processed = set()

if os.path.exists(OUTPUT):

    with open(OUTPUT, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            processed.add(row["path"])

    print("\n🔄 Existujúci CSV nájdený.")
    print("Už spracovaných:", len(processed))

else:

    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "dataset",
            "file",
            "path",
            "country",
            "gender",
            "speaker",
            "label",
            "class",
            "prediction",
            "score_spoof",
            "score_bonafide",
            "duration_sec",
            "correct"
        ])

    print("\n🆕 Vytvorený nový CSV.")

# ============================================================
# INFERENCE
# ============================================================

start = time.time()
new_count = 0
errors = 0

for i, (true_class, label, path) in enumerate(samples, 1):

    if path in processed:
        continue

    try:
        # Z cesty získame USA/UK, male/female a číslo osoby
        relative = os.path.relpath(path, ROOT)
        parts = relative.split(os.sep)

        country = parts[0]
        gender = parts[1]
        speaker = parts[2]

        # Všetko prevzorkujeme na 16 kHz mono
        audio, sr = librosa.load(
            path,
            sr=16000,
            mono=True
        )

        duration = len(audio) / sr

        with torch.inference_mode():
            result = pipe_df_arena(audio)

        prediction = result["label"]

        score_spoof = float(
            result["all_scores"]["spoof"]
        )

        score_bonafide = float(
            result["all_scores"]["bonafide"]
        )

        predicted_class = (
            "FAKE"
            if prediction == "spoof"
            else "REAL"
        )

        correct = int(
            predicted_class == true_class
        )

        # ====================================================
        # OKAMŽITÉ ULOŽENIE NA DRIVE
        # ====================================================

        with open(
            OUTPUT,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(f)

            writer.writerow([
                "Dataset2",
                os.path.basename(path),
                path,
                country,
                gender,
                speaker,
                label,
                true_class,
                prediction,
                score_spoof,
                score_bonafide,
                duration,
                correct
            ])

        new_count += 1

        print(
            f"[{i:02d}/{len(samples)}] "
            f"{country}/{gender}/{speaker} | "
            f"{os.path.basename(path)} | "
            f"{true_class} → {predicted_class} | "
            f"spoof={score_spoof:.4f} | "
            f"{duration:.1f}s"
        )

    except Exception as e:

        errors += 1

        print("\n❌ ERROR")
        print(path)
        print(type(e).__name__, str(e))

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 75)
print("HOTOVO")
print("=" * 75)

print("Nových spracovaných:", new_count)
print("Chyby:", errors)
print(
    "Čas:",
    round((time.time() - start) / 60, 2),
    "min"
)
print("Výsledky:", OUTPUT)

DF ARENA 1B — DATASET2
FAKE : 60
REAL : 20
TOTAL: 80
CSV  : /content/drive/MyDrive/df_arena_dataset2_scores.csv

🆕 Vytvorený nový CSV.
[01/80] UK/female/1 | synthetic_1.mp3 | FAKE → REAL | spoof=0.0646 | 48.1s
[02/80] UK/female/1 | synthetic_2.mp3 | FAKE → FAKE | spoof=0.8990 | 48.1s
[03/80] UK/female/1 | synthetic_3.mp3 | FAKE → FAKE | spoof=0.9927 | 48.1s
[04/80] UK/female/2 | synthetic_1.mp3 | FAKE → FAKE | spoof=0.7007 | 48.5s
[05/80] UK/female/2 | synthetic_2.mp3 | FAKE → FAKE | spoof=0.9971 | 48.5s
[06/80] UK/female/2 | synthetic_3.mp3 | FAKE → FAKE | spoof=0.9917 | 48.5s
[07/80] UK/female/3 | synthetic_1.mp3 | FAKE → FAKE | spoof=0.8224 | 44.1s
[08/80] UK/female/3 | synthetic_2.mp3 | FAKE → FAKE | spoof=0.5814 | 44.1s
[09/80] UK/female/3 | synthetic_3.mp3 | FAKE → FAKE | spoof=0.9977 | 44.1s
[10/80] UK/female/4 | synthetic_1.mp3 | FAKE → FAKE | spoof=0.9204 | 54.0s
[11/80] UK/female/4 | synthetic_2.mp3 | FAKE → FAKE | spoof=0.9804 | 54.0s
[12/80] UK/female/4 | synthetic_3.mp3 | 

/tmp/ipykernel_1273/2541380490.py:130: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[61/80] UK/female/1 | original.m4a | REAL → REAL | spoof=0.4621 | 48.0s


/tmp/ipykernel_1273/2541380490.py:130: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[62/80] UK/female/2 | original.m4a | REAL → FAKE | spoof=0.6720 | 48.4s


/tmp/ipykernel_1273/2541380490.py:130: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


... (11 outputs left out)

/tmp/ipykernel_1273/2541380490.py:130: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[79/80] USA/male/4 | original.m4a | REAL → FAKE | spoof=0.8767 | 63.5s


/tmp/ipykernel_1273/2541380490.py:130: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[80/80] USA/male/5 | original.m4a | REAL → FAKE | spoof=0.7130 | 60.5s

HOTOVO
Nových spracovaných: 80
Chyby: 0
Čas: 0.53 min
Výsledky: /content/drive/MyDrive/df_arena_dataset2_scores.csv


In [ ]:
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

CSV = "/content/drive/MyDrive/df_arena_dataset2_scores.csv"

df2 = pd.read_csv(CSV)

y_true = df2["label"].astype(int).values
score_fake = df2["score_spoof"].astype(float).values

# Rovnaký default threshold ako D1 a D3
y_pred = (score_fake >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
balanced_acc = balanced_accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, score_fake)

precision_fake = precision_score(
    y_true, y_pred, pos_label=1
)

recall_fake = recall_score(
    y_true, y_pred, pos_label=1
)

recall_real = recall_score(
    y_true, y_pred, pos_label=0
)

f1_fake = f1_score(
    y_true, y_pred, pos_label=1
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("=" * 75)
print("DF ARENA 1B V1 — DATASET2")
print("=" * 75)

print("Počet vzoriek:", len(df2))

print("\nHLAVNÉ METRIKY")
print("-" * 40)

print(f"Accuracy          : {accuracy*100:.2f}%")
print(f"Balanced Accuracy : {balanced_acc*100:.2f}%")
print(f"ROC AUC           : {auc:.4f}")

print("\nFAKE:")
print(f"Precision : {precision_fake*100:.2f}%")
print(f"Recall    : {recall_fake*100:.2f}%")
print(f"F1        : {f1_fake:.4f}")

print("\nREAL:")
print(f"Recall    : {recall_real*100:.2f}%")

print("\nCONFUSION MATRIX")
print("-" * 40)

print("                 PRED REAL    PRED FAKE")
print(f"TRUE REAL        {tn:9d}    {fp:9d}")
print(f"TRUE FAKE        {fn:9d}    {tp:9d}")

print("\nSprávne REAL :", tn, "/", tn + fp)
print("Falošne FAKE :", fp, "/", tn + fp)

print("Správne FAKE :", tp, "/", tp + fn)
print("Falošne REAL :", fn, "/", tp + fn)

print("\nSCORE_SPOOF DISTRIBÚCIA")
print("-" * 40)

for cls in ["REAL", "FAKE"]:

    scores = df2.loc[
        df2["class"] == cls,
        "score_spoof"
    ]

    print(f"\n{cls}")
    print(f"mean   : {scores.mean():.6f}")
    print(f"median : {scores.median():.6f}")
    print(f"min    : {scores.min():.6f}")
    print(f"max    : {scores.max():.6f}")

DF ARENA 1B V1 — DATASET2
Počet vzoriek: 80

HLAVNÉ METRIKY
----------------------------------------
Accuracy          : 87.50%
Balanced Accuracy : 78.33%
ROC AUC           : 0.9108

FAKE:
Precision : 87.88%
Recall    : 96.67%
F1        : 0.9206

REAL:
Recall    : 60.00%

CONFUSION MATRIX
----------------------------------------
                 PRED REAL    PRED FAKE
TRUE REAL               12            8
TRUE FAKE                2           58

Správne REAL : 12 / 20
Falošne FAKE : 8 / 20
Správne FAKE : 58 / 60
Falošne REAL : 2 / 60

SCORE_SPOOF DISTRIBÚCIA
----------------------------------------

REAL
mean   : 0.435294
median : 0.399082
min    : 0.036119
max    : 0.922539

FAKE
mean   : 0.877369
median : 0.923352
min    : 0.064603
max    : 0.997879


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# CSV S DF ARENA VÝSLEDKAMI
# ============================================================

FILES = {
    "Dataset1": "/content/drive/MyDrive/df_arena_dataset1_scores.csv",
    "Dataset2": "/content/drive/MyDrive/df_arena_dataset2_scores.csv",
    "Dataset3": "/content/drive/MyDrive/df_arena_dataset3_scores.csv",
}

# Starý model6 – výsledky, ktoré sme už namerali
MODEL6_AUC = {
    "Dataset1": 0.9184,
    "Dataset2": 0.6408,
    "Dataset3": 0.7821,
}

results = []

# ============================================================
# VYHODNOTENIE DF ARENA
# ============================================================

for dataset, path in FILES.items():

    df = pd.read_csv(path)

    y_true = df["label"].astype(int).values
    score = df["score_spoof"].astype(float).values

    # default threshold DF Arena
    y_pred = (score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)

    ba = balanced_accuracy_score(
        y_true,
        y_pred
    )

    auc = roc_auc_score(
        y_true,
        score
    )

    fake_precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    fake_recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    real_recall = recall_score(
        y_true,
        y_pred,
        pos_label=0,
        zero_division=0
    )

    fake_f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Dataset": dataset,
        "N": len(df),

        "DF_Arena_AUC": auc,
        "DF_Arena_Accuracy": acc,
        "DF_Arena_BA": ba,

        "FAKE_Precision": fake_precision,
        "FAKE_Recall": fake_recall,
        "REAL_Recall": real_recall,
        "FAKE_F1": fake_f1,

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,

        "Model6_AUC": MODEL6_AUC[dataset],

        "AUC_Improvement":
            auc - MODEL6_AUC[dataset]
    })


summary = pd.DataFrame(results)

# ============================================================
# VÝPIS
# ============================================================

print("=" * 100)
print("DF ARENA 1B V1 — FINÁLNE VÝSLEDKY")
print("=" * 100)

display_table = summary.copy()

percent_columns = [
    "DF_Arena_Accuracy",
    "DF_Arena_BA",
    "FAKE_Precision",
    "FAKE_Recall",
    "REAL_Recall",
]

for col in percent_columns:
    display_table[col] = (
        display_table[col] * 100
    ).round(2)

for col in [
    "DF_Arena_AUC",
    "FAKE_F1",
    "Model6_AUC",
    "AUC_Improvement"
]:
    display_table[col] = (
        display_table[col].round(4)
    )

print(
    display_table.to_string(index=False)
)


# ============================================================
# JEDNODUCHÉ POROVNANIE AUC
# ============================================================

print("\n" + "=" * 70)
print("MODEL6 vs DF ARENA — ROC AUC")
print("=" * 70)

for _, row in summary.iterrows():

    print(
        f"{row['Dataset']}: "
        f"Model6 = {row['Model6_AUC']:.4f} | "
        f"DF Arena = {row['DF_Arena_AUC']:.4f} | "
        f"Δ = {row['AUC_Improvement']:+.4f}"
    )


# ============================================================
# PRIEMERNÉ AUC CEZ 3 DATASETY
# ============================================================

mean_model6_auc = summary["Model6_AUC"].mean()
mean_df_auc = summary["DF_Arena_AUC"].mean()

print("\n" + "=" * 70)
print("PRIEMER CEZ 3 DATASETY")
print("=" * 70)

print(
    f"Model6 priemerné AUC : "
    f"{mean_model6_auc:.4f}"
)

print(
    f"DF Arena priemerné AUC: "
    f"{mean_df_auc:.4f}"
)

print(
    f"Rozdiel              : "
    f"{mean_df_auc - mean_model6_auc:+.4f}"
)


# ============================================================
# SPOJENÉ VŠETKY VZORKY
# ============================================================

all_dfs = []

for dataset, path in FILES.items():

    temp = pd.read_csv(path)
    temp["dataset_name"] = dataset

    all_dfs.append(temp)

combined = pd.concat(
    all_dfs,
    ignore_index=True
)

y_all = combined["label"].astype(int).values
score_all = combined["score_spoof"].astype(float).values
pred_all = (score_all >= 0.5).astype(int)

print("\n" + "=" * 70)
print("VŠETKY DF ARENA VZORKY SPOLU")
print("=" * 70)

print("N:", len(combined))

print(
    "Accuracy:",
    f"{accuracy_score(y_all, pred_all)*100:.2f}%"
)

print(
    "Balanced Accuracy:",
    f"{balanced_accuracy_score(y_all, pred_all)*100:.2f}%"
)

print(
    "ROC AUC:",
    f"{roc_auc_score(y_all, score_all):.4f}"
)

print(
    "FAKE Recall:",
    f"{recall_score(y_all, pred_all, pos_label=1)*100:.2f}%"
)

print(
    "REAL Recall:",
    f"{recall_score(y_all, pred_all, pos_label=0)*100:.2f}%"
)


# ============================================================
# ULOŽENIE SÚHRNU NA DRIVE
# ============================================================

SUMMARY_OUTPUT = (
    "/content/drive/MyDrive/"
    "df_arena_final_summary.csv"
)

summary.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

print("\n💾 Súhrn uložený:")
print(SUMMARY_OUTPUT)

DF ARENA 1B V1 — FINÁLNE VÝSLEDKY
 Dataset   N  DF_Arena_AUC  DF_Arena_Accuracy  DF_Arena_BA  FAKE_Precision  FAKE_Recall  REAL_Recall  FAKE_F1  TN  FP  FN  TP  Model6_AUC  AUC_Improvement
Dataset1  42        0.9138              64.29        64.29           87.50        33.33        95.24   0.4828  20   1  14   7      0.9184          -0.0046
Dataset2  80        0.9108              87.50        78.33           87.88        96.67        60.00   0.9206  12   8   2  58      0.6408           0.2700
Dataset3 854        0.9591              88.88        88.81           84.01        96.29        81.32   0.8973 344  79  16 415      0.7821           0.1770

MODEL6 vs DF ARENA — ROC AUC
Dataset1: Model6 = 0.9184 | DF Arena = 0.9138 | Δ = -0.0046
Dataset2: Model6 = 0.6408 | DF Arena = 0.9108 | Δ = +0.2700
Dataset3: Model6 = 0.7821 | DF Arena = 0.9591 | Δ = +0.1770

PRIEMER CEZ 3 DATASETY
Model6 priemerné AUC : 0.7804
DF Arena priemerné AUC: 0.9279
Rozdiel              : +0.1475

VŠETKY DF ARENA VZO

In [ ]:
import numpy as np
import librosa
import torch


SR = 16000

# približne podľa natívneho okna DF Arena
SEGMENT_SAMPLES = 64600
SEGMENT_SEC = SEGMENT_SAMPLES / SR

# 50 % overlap
HOP_SAMPLES = SEGMENT_SAMPLES // 2


def detect_segmented(path, threshold=0.5):

    # --------------------------------------------------------
    # AUDIO
    # --------------------------------------------------------

    audio, sr = librosa.load(
        path,
        sr=SR,
        mono=True
    )

    duration = len(audio) / SR

    # --------------------------------------------------------
    # SEGMENTY
    # --------------------------------------------------------

    segments = []

    # Ak je audio kratšie než jeden segment,
    # pošleme ho modelu celé.
    if len(audio) <= SEGMENT_SAMPLES:

        segments.append(audio)

    else:

        start = 0

        while start < len(audio):

            end = start + SEGMENT_SAMPLES
            segment = audio[start:end]

            # príliš krátky posledný zvyšok vynecháme
            if len(segment) < SR:
                break

            segments.append(segment)

            if end >= len(audio):
                break

            start += HOP_SAMPLES

    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    spoof_scores = []

    for segment in segments:

        with torch.inference_mode():

            result = pipe_df_arena(segment)

        spoof = float(
            result["all_scores"]["spoof"]
        )

        spoof_scores.append(spoof)

    spoof_scores = np.array(spoof_scores)

    # --------------------------------------------------------
    # AGREGÁCIA
    # --------------------------------------------------------

    mean_score = float(
        np.mean(spoof_scores)
    )

    median_score = float(
        np.median(spoof_scores)
    )

    max_score = float(
        np.max(spoof_scores)
    )

    p90_score = float(
        np.percentile(spoof_scores, 90)
    )

    fake_ratio = float(
        np.mean(spoof_scores >= threshold)
    )

    # Zatiaľ iba pracovné rozhodnutie.
    prediction = (
        "FAKE"
        if mean_score >= threshold
        else "REAL"
    )

    return {

        "prediction": prediction,

        "score": mean_score,

        "mean_score": mean_score,
        "median_score": median_score,
        "max_score": max_score,
        "p90_score": p90_score,

        "fake_ratio": fake_ratio,

        "segments": len(spoof_scores),

        "duration_sec": duration,

        "segment_scores": spoof_scores
    }


print("Segmentovaný detektor pripravený.")

print(
    f"Dĺžka segmentu: {SEGMENT_SEC:.3f} s"
)

print(
    f"Posun: {HOP_SAMPLES / SR:.3f} s"
)

Segmentovaný detektor pripravený.
Dĺžka segmentu: 4.037 s
Posun: 2.019 s


In [ ]:
import pandas as pd
import os


tests = [

    # Dataset1 - FAKE, ktoré whole-file model zle označil
    (
        "D1 FAKE",
        "/content/ai_voice_project/Dataset1/Fake/Fake/Recording (1).wav"
    ),

    (
        "D1 FAKE",
        "/content/ai_voice_project/Dataset1/Fake/Fake/Recording (7).wav"
    ),

    # Dataset3 - REAL, ktorý bol zle označený
    (
        "D3 REAL",
        "/content/ai_voice_project/Dataset3/Real/real_038.wav"
    ),

]


for expected, path in tests:

    print("\n" + "=" * 75)

    print("TEST:", expected)
    print("FILE:", os.path.basename(path))

    if not os.path.exists(path):

        print("❌ Súbor sa na tejto ceste nenašiel.")
        continue

    result = detect_segmented(
        path,
        threshold=0.5
    )

    print(
        f"Duration : {result['duration_sec']:.2f} s"
    )

    print(
        f"Segments : {result['segments']}"
    )

    print(
        f"Mean     : {result['mean_score']:.4f}"
    )

    print(
        f"Median   : {result['median_score']:.4f}"
    )

    print(
        f"P90      : {result['p90_score']:.4f}"
    )

    print(
        f"Max      : {result['max_score']:.4f}"
    )

    print(
        f"FAKE ratio: {result['fake_ratio']*100:.1f}%"
    )

    print(
        "Prediction:",
        result["prediction"]
    )

    print(
        "\nPrvých 20 segment scores:"
    )

    print(
        np.round(
            result["segment_scores"][:20],
            4
        )
    )


TEST: D1 FAKE
FILE: Recording (1).wav
Duration : 494.94 s
Segments : 245
Mean     : 0.1412
Median   : 0.0749
P90      : 0.3099
Max      : 0.8935
FAKE ratio: 5.3%
Prediction: REAL

Prvých 20 segment scores:
[0.2689 0.0883 0.4703 0.2669 0.2312 0.1811 0.0866 0.0609 0.0376 0.0575
 0.0527 0.0342 0.2287 0.1518 0.1148 0.1214 0.0274 0.0552 0.6573 0.073 ]

TEST: D1 FAKE
FILE: Recording (7).wav
Duration : 324.18 s
Segments : 160
Mean     : 0.1939
Median   : 0.1417
P90      : 0.4372
Max      : 0.8478
FAKE ratio: 6.9%
Prediction: REAL

Prvých 20 segment scores:
[0.0832 0.0179 0.0396 0.2967 0.1385 0.3511 0.2871 0.3805 0.0949 0.1591
 0.0983 0.0957 0.0277 0.1095 0.1532 0.1651 0.0471 0.0702 0.1056 0.0872]

TEST: D3 REAL
FILE: real_038.wav
❌ Súbor sa na tejto ceste nenašiel.


In [ ]:
import pandas as pd
import os

# Načítame už existujúce whole-file výsledky
d1 = pd.read_csv(
    "/content/drive/MyDrive/df_arena_dataset1_scores.csv"
)

d3 = pd.read_csv(
    "/content/drive/MyDrive/df_arena_dataset3_scores.csv"
)

tests = []

# ------------------------------------------------------------
# D1: 2 FAKE, ktoré whole-file model označil ako REAL
# ------------------------------------------------------------

d1_bad_fake = (
    d1[
        (d1["class"] == "FAKE") &
        (d1["score_spoof"] < 0.5)
    ]
    .sort_values("score_spoof")
    .head(2)
)

for _, row in d1_bad_fake.iterrows():
    tests.append(
        ("D1 FAKE - whole-file zle", row["path"])
    )


# ------------------------------------------------------------
# D1: 2 FAKE, ktoré whole-file model označil správne
# ------------------------------------------------------------

d1_good_fake = (
    d1[
        (d1["class"] == "FAKE") &
        (d1["score_spoof"] >= 0.5)
    ]
    .sort_values("score_spoof", ascending=False)
    .head(2)
)

for _, row in d1_good_fake.iterrows():
    tests.append(
        ("D1 FAKE - whole-file správne", row["path"])
    )


# ------------------------------------------------------------
# D1: 2 REAL
# ------------------------------------------------------------

d1_real = (
    d1[d1["class"] == "REAL"]
    .sort_values("score_spoof")
    .head(2)
)

for _, row in d1_real.iterrows():
    tests.append(
        ("D1 REAL", row["path"])
    )


# ------------------------------------------------------------
# D3: najhoršie REAL -> FAKE prípady
# ------------------------------------------------------------

d3_bad_real = (
    d3[
        (d3["class"] == "REAL") &
        (d3["score_spoof"] >= 0.5)
    ]
    .sort_values("score_spoof", ascending=False)
    .head(3)
)

for _, row in d3_bad_real.iterrows():
    tests.append(
        ("D3 REAL - whole-file zle", row["path"])
    )


# ------------------------------------------------------------
# KONTROLA
# ------------------------------------------------------------

print("=" * 90)
print("TESTOVACIE SÚBORY")
print("=" * 90)

for expected, path in tests:

    print()
    print(expected)
    print(os.path.basename(path))
    print("existuje:", os.path.exists(path))
    print(path)

TESTOVACIE SÚBORY

D1 FAKE - whole-file zle
Recording (7).wav
existuje: True
/content/ai_voice_project/Dataset1/Fake/Fake/Recording (7).wav

D1 FAKE - whole-file zle
Recording (11).wav
existuje: True
/content/ai_voice_project/Dataset1/Fake/Fake/Recording (11).wav

D1 FAKE - whole-file správne
Recording (16).wav
existuje: True
/content/ai_voice_project/Dataset1/Fake/Fake/Recording (16).wav

D1 FAKE - whole-file správne
Recording (8).wav
existuje: True
/content/ai_voice_project/Dataset1/Fake/Fake/Recording (8).wav

D1 REAL
Recording (6).wav
existuje: True
/content/ai_voice_project/Dataset1/Real/Real/Recording (6).wav

D1 REAL
Recording (14).wav
existuje: True
/content/ai_voice_project/Dataset1/Real/Real/Recording (14).wav

D3 REAL - whole-file zle
real_038.wav
existuje: True
/content/ai_voice_project/Dataset3/Real_Audios/real_038.wav

D3 REAL - whole-file zle
real_093.wav
existuje: True
/content/ai_voice_project/Dataset3/Real_Audios/real_093.wav

D3 REAL - whole-file zle
real_019.wav
exi

In [ ]:
import numpy as np
import pandas as pd
import os

rows = []

print("=" * 100)
print("WHOLE-FILE vs SEGMENTOVANÝ DF ARENA")
print("=" * 100)

for i, (expected, path) in enumerate(tests, 1):

    print(
        f"\n[{i}/{len(tests)}] "
        f"{expected} | {os.path.basename(path)}"
    )

    result = detect_segmented(
        path,
        threshold=0.5
    )

    # Nájdeme pôvodné whole-file skóre
    if expected.startswith("D1"):
        source_df = d1
    else:
        source_df = d3

    original_row = source_df[
        source_df["path"] == path
    ].iloc[0]

    whole_score = float(
        original_row["score_spoof"]
    )

    rows.append({
        "test": expected,
        "file": os.path.basename(path),
        "whole": whole_score,
        "mean": result["mean_score"],
        "median": result["median_score"],
        "p90": result["p90_score"],
        "max": result["max_score"],
        "fake_ratio": result["fake_ratio"],
        "segments": result["segments"],
    })

    print(
        f"whole={whole_score:.4f} | "
        f"mean={result['mean_score']:.4f} | "
        f"median={result['median_score']:.4f} | "
        f"p90={result['p90_score']:.4f} | "
        f"max={result['max_score']:.4f} | "
        f"fake_ratio={result['fake_ratio']*100:.1f}%"
    )


comparison = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("SÚHRN")
print("=" * 100)

print(
    comparison.round(4).to_string(index=False)
)

WHOLE-FILE vs SEGMENTOVANÝ DF ARENA

[1/9] D1 FAKE - whole-file zle | Recording (7).wav
whole=0.0832 | mean=0.1939 | median=0.1417 | p90=0.4372 | max=0.8478 | fake_ratio=6.9%

[2/9] D1 FAKE - whole-file zle | Recording (11).wav
whole=0.1881 | mean=0.1542 | median=0.0973 | p90=0.3420 | max=0.8862 | fake_ratio=7.2%

[3/9] D1 FAKE - whole-file správne | Recording (16).wav
whole=0.8281 | mean=0.1734 | median=0.1026 | p90=0.4371 | max=0.8561 | fake_ratio=6.7%

[4/9] D1 FAKE - whole-file správne | Recording (8).wav
whole=0.8124 | mean=0.2654 | median=0.2053 | p90=0.6051 | max=0.9843 | fake_ratio=15.7%

[5/9] D1 REAL | Recording (6).wav
whole=0.0120 | mean=0.0138 | median=0.0117 | p90=0.0186 | max=0.1490 | fake_ratio=0.0%

[6/9] D1 REAL | Recording (14).wav
whole=0.0137 | mean=0.0171 | median=0.0134 | p90=0.0285 | max=0.1081 | fake_ratio=0.0%

[7/9] D3 REAL - whole-file zle | real_038.wav
whole=0.9998 | mean=0.9990 | median=0.9994 | p90=0.9997 | max=0.9998 | fake_ratio=100.0%

[8/9] D3 REAL -

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    recall_score
)

FILES = {
    "Dataset1": "/content/drive/MyDrive/df_arena_dataset1_scores.csv",
    "Dataset2": "/content/drive/MyDrive/df_arena_dataset2_scores.csv",
    "Dataset3": "/content/drive/MyDrive/df_arena_dataset3_scores.csv",
}

datasets = {}

for name, path in FILES.items():

    df = pd.read_csv(path)

    datasets[name] = {
        "y": df["label"].astype(int).values,
        "score": df["score_spoof"].astype(float).values
    }


# ============================================================
# TESTOVANÉ THRESHOLDY
# ============================================================

thresholds = np.arange(
    0.05,
    0.951,
    0.005
)

rows = []

for threshold in thresholds:

    dataset_BAs = []

    all_y = []
    all_scores = []

    for name, data in datasets.items():

        y = data["y"]
        score = data["score"]

        pred = (
            score >= threshold
        ).astype(int)

        ba = balanced_accuracy_score(
            y,
            pred
        )

        dataset_BAs.append(ba)

        all_y.extend(y)
        all_scores.extend(score)

    all_y = np.array(all_y)
    all_scores = np.array(all_scores)

    all_pred = (
        all_scores >= threshold
    ).astype(int)

    rows.append({

        "threshold": threshold,

        # každý dataset má rovnakú váhu
        "mean_dataset_BA":
            np.mean(dataset_BAs),

        # všetkých 976 vzoriek spolu
        "combined_BA":
            balanced_accuracy_score(
                all_y,
                all_pred
            ),

        "combined_accuracy":
            accuracy_score(
                all_y,
                all_pred
            ),

        "fake_recall":
            recall_score(
                all_y,
                all_pred,
                pos_label=1
            ),

        "real_recall":
            recall_score(
                all_y,
                all_pred,
                pos_label=0
            )
    })


threshold_results = pd.DataFrame(rows)


# ============================================================
# NAJLEPŠÍ THRESHOLD
# Každý dataset má rovnakú váhu
# ============================================================

best_idx = (
    threshold_results[
        "mean_dataset_BA"
    ].idxmax()
)

best = threshold_results.loc[
    best_idx
]

best_threshold = float(
    best["threshold"]
)


print("=" * 80)
print("THRESHOLD ANALÝZA")
print("=" * 80)

print(
    f"\nNajlepší development threshold: "
    f"{best_threshold:.3f}"
)

print(
    f"Priemerná dataset BA: "
    f"{best['mean_dataset_BA']*100:.2f}%"
)

print(
    f"Combined BA: "
    f"{best['combined_BA']*100:.2f}%"
)

print(
    f"Combined Accuracy: "
    f"{best['combined_accuracy']*100:.2f}%"
)

print(
    f"FAKE Recall: "
    f"{best['fake_recall']*100:.2f}%"
)

print(
    f"REAL Recall: "
    f"{best['real_recall']*100:.2f}%"
)


# ============================================================
# DEFAULT 0.5 PRE POROVNANIE
# ============================================================

default_row = threshold_results.iloc[
    (
        threshold_results["threshold"]
        - 0.5
    ).abs().argsort()[:1]
]

print("\n" + "=" * 80)
print("DEFAULT THRESHOLD 0.500")
print("=" * 80)

print(
    default_row[
        [
            "threshold",
            "mean_dataset_BA",
            "combined_BA",
            "combined_accuracy",
            "fake_recall",
            "real_recall"
        ]
    ].to_string(index=False)
)


# ============================================================
# VÝSLEDKY NA KAŽDOM DATASETE PRI NOVOM THRESHOLDE
# ============================================================

print("\n" + "=" * 80)
print(
    f"VÝSLEDKY PRI THRESHOLDE "
    f"{best_threshold:.3f}"
)
print("=" * 80)

for name, data in datasets.items():

    y = data["y"]
    score = data["score"]

    pred = (
        score >= best_threshold
    ).astype(int)

    ba = balanced_accuracy_score(
        y,
        pred
    )

    acc = accuracy_score(
        y,
        pred
    )

    fake_recall = recall_score(
        y,
        pred,
        pos_label=1
    )

    real_recall = recall_score(
        y,
        pred,
        pos_label=0
    )

    print(f"\n{name}")

    print(
        f"Accuracy   : {acc*100:.2f}%"
    )

    print(
        f"BA         : {ba*100:.2f}%"
    )

    print(
        f"FAKE Recall: {fake_recall*100:.2f}%"
    )

    print(
        f"REAL Recall: {real_recall*100:.2f}%"
    )


# ============================================================
# TOP 10 THRESHOLDOV
# ============================================================

print("\n" + "=" * 80)
print("TOP 10 THRESHOLDOV")
print("=" * 80)

top10 = (
    threshold_results
    .sort_values(
        "mean_dataset_BA",
        ascending=False
    )
    .head(10)
)

print(
    top10.round(4).to_string(
        index=False
    )
)

THRESHOLD ANALÝZA

Najlepší development threshold: 0.360
Priemerná dataset BA: 80.12%
Combined BA: 84.74%
Combined Accuracy: 85.35%
FAKE Recall: 97.07%
REAL Recall: 72.41%

DEFAULT THRESHOLD 0.500
 threshold  mean_dataset_BA  combined_BA  combined_accuracy  fake_recall  real_recall
       0.5         0.771416     0.873922           0.877049       0.9375     0.810345

VÝSLEDKY PRI THRESHOLDE 0.360

Dataset1
Accuracy   : 83.33%
BA         : 83.33%
FAKE Recall: 76.19%
REAL Recall: 90.48%

Dataset2
Accuracy   : 85.00%
BA         : 71.67%
FAKE Recall: 98.33%
REAL Recall: 45.00%

Dataset3
Accuracy   : 85.48%
BA         : 85.36%
FAKE Recall: 97.91%
REAL Recall: 72.81%

TOP 10 THRESHOLDOV
 threshold  mean_dataset_BA  combined_BA  combined_accuracy  fake_recall  real_recall
     0.360           0.8012       0.8474             0.8535       0.9707       0.7241
     0.355           0.8004       0.8453             0.8514       0.9707       0.7198
     0.400           0.7976       0.8573            